In [1]:
"""
MILK10k 2025 Challenge - Complete Pipeline (CORRECTED v3)
Fixes:
  - A.Flip → A.HorizontalFlip + A.VerticalFlip
  - A.GaussNoise var_limit → std_range (albumentations >=2.0)
  - A.CoarseDropout → A.CoarseDropout (num_holes_range / hole_height_range API)
  - A.ShiftScaleRotate p kept, no API change needed
  - OrdinalEncoder unknown handling
  - DataParallel + T4x2 memory tuning
"""

import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import f1_score
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import albumentations

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================

class Config:
    BASE_DIR          = "/kaggle/input/datasets/hamzabinbutt/isic-challenge-datasets"
    TRAIN_IMG_DIR     = f"{BASE_DIR}/MILK10k_Training_Input/MILK10k_Training_Input"
    TEST_IMG_DIR      = f"{BASE_DIR}/MILK10k_Test_Input/MILK10k_Test_Input"
    TRAIN_METADATA    = f"{BASE_DIR}/MILK10k_Training_Metadata.csv"
    TEST_METADATA     = f"{BASE_DIR}/MILK10k_Test_Metadata.csv"
    TRAIN_GROUNDTRUTH = f"{BASE_DIR}/MILK10k_Training_GroundTruth.csv"
    TRAIN_SUPPLEMENT  = f"{BASE_DIR}/MILK10k_Training_Supplement.csv"
    OUTPUT_DIR        = "/kaggle/working"

    MODEL_NAME  = "tf_efficientnetv2_m"
    IMG_SIZE    = 384
    NUM_CLASSES = 11

    N_FOLDS               = 3
    EPOCHS                = 12
    BATCH_SIZE            = 12    # per DataParallel call; safe for T4x2
    NUM_WORKERS           = 4
    LEARNING_RATE         = 3e-4
    WEIGHT_DECAY          = 1e-5
    GRADIENT_ACCUMULATION = 2
    MAX_GRAD_NORM         = 1.0

    FOCAL_ALPHA  = 0.25
    FOCAL_GAMMA  = 2.0
    DICE_WEIGHT  = 0.6
    T_0          = 10
    SEED         = 42
    USE_AMP      = True

    CLASSES = ['AKIEC', 'BCC', 'BEN_OTH', 'BKL', 'DF', 'INF',
               'MAL_OTH', 'MEL', 'NV', 'SCCKA', 'VASC']

    MONET_COLS = [
        'MONET_ulceration_crust',
        'MONET_hair',
        'MONET_vasculature_vessels',
        'MONET_erythema',
        'MONET_pigmented',
        'MONET_gel_water_drop_fluid_dermoscopy_liquid',
        'MONET_skin_markings_pen_ink_purple_pen',
    ]
    N_MONET = 7

# ============================================================
# UTILITIES
# ============================================================

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def save_checkpoint(state_dict, fold, score, output_dir):
    path = f"{output_dir}/best_model_fold{fold}.pth"
    torch.save({'model_state_dict': state_dict, 'fold': fold, 'score': score}, path)
    print(f"  ✓ Checkpoint saved fold{fold}  F1={score:.4f}")

# ============================================================
# ALBUMENTATIONS VERSION-SAFE AUGMENTATION
# ============================================================

def _albu_version():
    return tuple(int(x) for x in albumentations.__version__.split(".")[:2])

def get_train_transforms(img_size=384):
    ver = _albu_version()

    # GaussNoise API changed in v2.0: var_limit → std_range (takes std, not variance)
    if ver >= (2, 0):
        noise = A.GaussNoise(std_range=(0.03, 0.09), p=1)          # ~var 10-50 equiv
        dropout = A.CoarseDropout(
            num_holes_range=(1, 6),
            hole_height_range=(8, 24),
            hole_width_range=(8, 24),
            fill=0, p=0.3
        )
    else:
        noise = A.GaussNoise(var_limit=(10.0, 50.0), p=1)
        dropout = A.CoarseDropout(
            max_holes=6, max_height=24, max_width=24,
            min_holes=1, fill_value=0, p=0.3
        )

    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30, p=0.6),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=15, p=0.6),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.6),
        A.OneOf([
            noise,
            A.GaussianBlur(blur_limit=(3, 5), p=1),
        ], p=0.3),
        dropout,
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_valid_transforms(img_size=384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

# ============================================================
# DATA LOADING
# ============================================================

_site_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

def load_and_merge_data():
    print("Loading CSVs...")
    metadata    = pd.read_csv(Config.TRAIN_METADATA)
    groundtruth = pd.read_csv(Config.TRAIN_GROUNDTRUTH)
    supplement  = pd.read_csv(Config.TRAIN_SUPPLEMENT)

    print(f"  metadata {metadata.shape}  groundtruth {groundtruth.shape}  supplement {supplement.shape}")

    # supplement key is isic_id, not lesion_id
    meta_supp = metadata.merge(supplement, on='isic_id', how='left')
    # deduplicate: 2 rows per lesion (clinical + dermoscopic) → keep first
    meta_unique = meta_supp.drop_duplicates(subset='lesion_id', keep='first').reset_index(drop=True)
    train_df = meta_unique.merge(groundtruth, on='lesion_id', how='inner')

    print(f"  merged train_df: {train_df.shape}")
    return train_df

def load_test_data():
    test_df = pd.read_csv(Config.TEST_METADATA)
    test_df = test_df.drop_duplicates(subset='lesion_id', keep='first').reset_index(drop=True)
    print(f"  test_df (deduped): {test_df.shape}")
    return test_df

def preprocess_metadata(train_df, test_df=None):
    print("\nPreprocessing metadata...")

    def encode(df, fit=False):
        df = df.copy()
        df['age_encoded']       = (df['age_approx'].fillna(40) / 5).astype(int).clip(0, 19)
        df['sex_encoded']       = df['sex'].str.lower().map({'male': 0, 'female': 1}).fillna(0).astype(int)
        df['skin_tone_encoded'] = df['skin_tone_class'].fillna(3).astype(int).clip(0, 5)
        site_vals = df[['site']].fillna('unknown')
        if fit:
            _site_encoder.fit(site_vals)
        enc = _site_encoder.transform(site_vals).astype(int).flatten()
        df['site_encoded'] = np.where(enc < 0, 0, enc + 1)
        for col in Config.MONET_COLS:
            df[col] = df[col].fillna(0.0).astype(float) if col in df.columns else 0.0
        return df

    train_df = encode(train_df, fit=True)
    n_sites  = int(train_df['site_encoded'].max()) + 2
    if test_df is not None:
        test_df = encode(test_df, fit=False)
    print(f"  n_sites embedding: {n_sites}")
    return train_df, test_df, n_sites

# ============================================================
# IMAGE PATHS
# ============================================================

def get_image_paths(lesion_id, base_dir):
    lesion_dir = Path(base_dir) / lesion_id
    if not lesion_dir.exists():
        raise FileNotFoundError(f"Not found: {lesion_dir}")
    imgs = sorted(list(lesion_dir.glob("*.jpg")) +
                  list(lesion_dir.glob("*.JPG")) +
                  list(lesion_dir.glob("*.jpeg")))
    if len(imgs) < 2:
        raise ValueError(f"Need 2 images in {lesion_dir}, got {len(imgs)}: {imgs}")
    return str(imgs[0]), str(imgs[1])

# ============================================================
# DATASET
# ============================================================

class MILKDataset(Dataset):
    def __init__(self, df, base_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.base_dir = base_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        lesion_id = row['lesion_id']
        try:
            p1, p2 = get_image_paths(lesion_id, self.base_dir)
            img1 = cv2.cvtColor(cv2.imread(p1), cv2.COLOR_BGR2RGB)
            img2 = cv2.cvtColor(cv2.imread(p2), cv2.COLOR_BGR2RGB)
            if self.transform:
                img1 = self.transform(image=img1)['image']
                img2 = self.transform(image=img2)['image']

            monet = torch.tensor([float(row[c]) for c in Config.MONET_COLS], dtype=torch.float32)
            labels = (torch.tensor([float(row[c]) for c in Config.CLASSES], dtype=torch.float32)
                      if not self.is_test else torch.zeros(Config.NUM_CLASSES))

            return {
                'img1':      img1,
                'img2':      img2,
                'age':       torch.tensor(int(row['age_encoded']),       dtype=torch.long),
                'sex':       torch.tensor(int(row['sex_encoded']),       dtype=torch.long),
                'skin_tone': torch.tensor(int(row['skin_tone_encoded']), dtype=torch.long),
                'site':      torch.tensor(int(row['site_encoded']),      dtype=torch.long),
                'monet':     monet,
                'labels':    labels,
                'lesion_id': lesion_id,
            }
        except Exception as e:
            print(f"  [WARN] {lesion_id}: {e}")
            return self.__getitem__((idx + 1) % len(self))

# ============================================================
# MODEL
# ============================================================

class DualStreamModel(nn.Module):
    def __init__(self, model_name='tf_efficientnetv2_m', num_classes=11,
                 pretrained=True, n_sites=35):
        super().__init__()
        self.encoder  = timm.create_model(model_name, pretrained=pretrained,
                                          num_classes=0, global_pool='')
        feat_dim      = self.encoder.num_features
        self.pool     = nn.AdaptiveAvgPool2d(1)

        self.age_embed       = nn.Embedding(20,      16)
        self.sex_embed       = nn.Embedding(2,        8)
        self.skin_tone_embed = nn.Embedding(6,        8)
        self.site_embed      = nn.Embedding(n_sites, 32)
        self.monet_fc        = nn.Sequential(nn.Linear(Config.N_MONET, 32), nn.ReLU(), nn.Dropout(0.2))

        fusion_in = feat_dim * 2 + 96   # 96 = 16+8+8+32+32
        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 256),       nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.3),
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, img1, img2, age, sex, skin_tone, site, monet):
        f1 = self.pool(self.encoder(img1)).flatten(1)
        f2 = self.pool(self.encoder(img2)).flatten(1)
        meta = torch.cat([self.age_embed(age), self.sex_embed(sex),
                          self.skin_tone_embed(skin_tone), self.site_embed(site),
                          self.monet_fc(monet)], dim=1)
        return self.classifier(self.fusion(torch.cat([f1, f2, meta], dim=1)))

# ============================================================
# LOSS
# ============================================================

class FocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, dice_weight=0.6):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt    = torch.where(targets == 1, probs, 1 - probs)
        focal = (self.alpha * (1 - pt) ** self.gamma * bce).mean()
        s     = 1e-6
        dice  = torch.stack([1 - (2*(probs[:,i]*targets[:,i]).sum() + s) /
                              (probs[:,i].sum() + targets[:,i].sum() + s)
                             for i in range(targets.shape[1])]).mean()
        return (1 - self.dice_weight) * focal + self.dice_weight * dice

# ============================================================
# METRICS
# ============================================================

def macro_f1(y_true, y_pred, threshold=0.5):
    y_bin  = (y_pred >= threshold).astype(int)
    scores = [f1_score(y_true[:,i], y_bin[:,i], zero_division=0) for i in range(y_true.shape[1])]
    return float(np.mean(scores)), scores

# ============================================================
# TRAIN / VALIDATE / PREDICT
# ============================================================

def run_batch(model, batch, device):
    return model(
        batch['img1'].to(device, non_blocking=True),
        batch['img2'].to(device, non_blocking=True),
        batch['age'].to(device, non_blocking=True),
        batch['sex'].to(device, non_blocking=True),
        batch['skin_tone'].to(device, non_blocking=True),
        batch['site'].to(device, non_blocking=True),
        batch['monet'].to(device, non_blocking=True),
    )

def train_one_epoch(model, loader, criterion, optimizer, device, scaler, epoch):
    model.train(); total_loss = 0.0; optimizer.zero_grad()
    pbar = tqdm(enumerate(loader), total=len(loader), desc=f'Epoch {epoch+1} Train')
    for step, batch in pbar:
        labels = batch['labels'].to(device, non_blocking=True)
        with autocast(enabled=Config.USE_AMP):
            logits = run_batch(model, batch, device)
            loss   = criterion(logits, labels) / Config.GRADIENT_ACCUMULATION
        scaler.scale(loss).backward()
        if (step + 1) % Config.GRADIENT_ACCUMULATION == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        total_loss += loss.item() * Config.GRADIENT_ACCUMULATION
        pbar.set_postfix(loss=f"{total_loss/(step+1):.4f}")
    return total_loss / len(loader)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval(); total_loss = 0.0; all_p, all_l = [], []
    for batch in tqdm(loader, desc='Validate'):
        labels = batch['labels'].to(device, non_blocking=True)
        logits = run_batch(model, batch, device)
        total_loss += criterion(logits, labels).item()
        all_p.append(torch.sigmoid(logits).cpu().numpy())
        all_l.append(labels.cpu().numpy())
    all_p = np.vstack(all_p); all_l = np.vstack(all_l)
    mf1, cf1 = macro_f1(all_l, all_p)
    return total_loss / len(loader), mf1, cf1

@torch.no_grad()
def predict(model, loader, device):
    model.eval(); all_p, all_ids = [], []
    for batch in tqdm(loader, desc='Predict'):
        logits = run_batch(model, batch, device)
        all_p.append(torch.sigmoid(logits).cpu().numpy())
        all_ids.extend(batch['lesion_id'])
    return all_ids, np.vstack(all_p)

# ============================================================
# FOLD TRAINING
# ============================================================
def train_fold(fold, train_df, n_sites, device):
    print(f"\n{'='*60}\n  FOLD {fold+1}/{Config.N_FOLDS}\n{'='*60}")
    skf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)
    strat = train_df[Config.CLASSES].idxmax(axis=1)
    for fi, (tr_idx, va_idx) in enumerate(skf.split(train_df, strat)):
        if fi == fold: break

    tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
    va_df = train_df.iloc[va_idx].reset_index(drop=True)
    print(f"  train={len(tr_df)}  val={len(va_df)}")

    tr_ds = MILKDataset(tr_df, Config.TRAIN_IMG_DIR, get_train_transforms(Config.IMG_SIZE))
    va_ds = MILKDataset(va_df, Config.TRAIN_IMG_DIR, get_valid_transforms(Config.IMG_SIZE))

    # ← NEW: weighted sampler for rare classes
    from torch.utils.data import WeightedRandomSampler
    class_counts = np.array([train_df[c].sum() for c in Config.CLASSES])
    sample_weights = []
    for _, row in tr_df.iterrows():
        label_idx = [i for i, c in enumerate(Config.CLASSES) if row[c] == 1]
        w = max([1.0 / class_counts[i] for i in label_idx]) if label_idx else 1.0
        sample_weights.append(w)
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    tr_loader = DataLoader(tr_ds, batch_size=Config.BATCH_SIZE, sampler=sampler,
                           num_workers=Config.NUM_WORKERS, pin_memory=True, drop_last=True)
    va_loader = DataLoader(va_ds, batch_size=Config.BATCH_SIZE * 2, shuffle=False,
                           num_workers=Config.NUM_WORKERS, pin_memory=True)

    model = DualStreamModel(Config.MODEL_NAME, Config.NUM_CLASSES, pretrained=True, n_sites=n_sites)
    n_gpus = torch.cuda.device_count()
    if n_gpus > 1:
        print(f"  DataParallel across {n_gpus} GPUs")
        model = nn.DataParallel(model)
    model = model.to(device)

    criterion = FocalDiceLoss(Config.FOCAL_ALPHA, Config.FOCAL_GAMMA, Config.DICE_WEIGHT)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE,
                                  weight_decay=Config.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=Config.T_0, T_mult=1, eta_min=1e-6)
    scaler = GradScaler(enabled=Config.USE_AMP)

    best_f1 = 0.0
    for epoch in range(Config.EPOCHS):
        tr_loss = train_one_epoch(model, tr_loader, criterion, optimizer, device, scaler, epoch)
        va_loss, va_f1, cf1 = validate(model, va_loader, criterion, device)
        scheduler.step()
        print(f"  Ep {epoch+1:2d}  tr={tr_loss:.4f}  va={va_loss:.4f}  F1={va_f1:.4f}")
        print(f"  {dict(zip(Config.CLASSES, [f'{s:.3f}' for s in cf1]))}")
        if va_f1 > best_f1:
            best_f1 = va_f1
            raw = model.module if hasattr(model, 'module') else model
            save_checkpoint(raw.state_dict(), fold, va_f1, Config.OUTPUT_DIR)

    print(f"\n  Fold {fold+1} best F1: {best_f1:.4f}")
    del model, optimizer, scheduler, tr_loader, va_loader, tr_ds, va_ds
    gc.collect(); torch.cuda.empty_cache()
    return best_f1

# ============================================================
# MAIN
# ============================================================

def main():
    print("=" * 70)
    print(f"MILK10k 2025  |  albumentations {albumentations.__version__}")
    print("=" * 70)
    set_seed(Config.SEED)
    device = get_device()
    print(f"Device: {device}")
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

    print("\n" + "="*70)
    train_df = load_and_merge_data()
    test_df  = load_test_data()
    train_df, test_df, n_sites = preprocess_metadata(train_df, test_df)

    print(f"\nTrain: {len(train_df)}  Test: {len(test_df)}")
    print("Class distribution:")
    for cls in Config.CLASSES:
        n = int(train_df[cls].sum())
        print(f"  {cls:10s}: {n:5d}  ({n/len(train_df)*100:.1f}%)")

    print("\n" + "="*70 + "\nCross-Validation\n" + "="*70)
    fold_scores = []
    for fold in range(Config.N_FOLDS):
        fold_scores.append(train_fold(fold, train_df, n_sites, device))

    print("\nCV Summary:")
    for i, s in enumerate(fold_scores): print(f"  Fold {i+1}: {s:.4f}")
    print(f"  Mean : {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

    # ---- Inference ----
    print("\n" + "="*70 + "\nTest Inference\n" + "="*70)
    te_ds = MILKDataset(test_df, Config.TEST_IMG_DIR,
                        get_valid_transforms(Config.IMG_SIZE), is_test=True)
    te_loader = DataLoader(te_ds, batch_size=Config.BATCH_SIZE * 2, shuffle=False,
                           num_workers=Config.NUM_WORKERS, pin_memory=True)

    all_fold_preds = []; lesion_ids_out = None
    for fold in range(Config.N_FOLDS):
        print(f"  Loading fold {fold+1}...")
        model = DualStreamModel(Config.MODEL_NAME, Config.NUM_CLASSES,
                                pretrained=False, n_sites=n_sites).to(device)
        ckpt = torch.load(f"{Config.OUTPUT_DIR}/best_model_fold{fold}.pth", map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        ids, preds = predict(model, te_loader, device)
        all_fold_preds.append(preds)
        if lesion_ids_out is None: lesion_ids_out = ids
        del model; gc.collect(); torch.cuda.empty_cache()

    final_preds = np.mean(all_fold_preds, axis=0).clip(0, 1)
    submission  = pd.DataFrame({'lesion': lesion_ids_out,
                                **{c: final_preds[:, i] for i, c in enumerate(Config.CLASSES)}})
    out_path = f"{Config.OUTPUT_DIR}/submission.csv"
    submission.to_csv(out_path, index=False)
    print(f"\nSubmission → {out_path}  shape={submission.shape}")
    print(submission.head())
    print(f"\nDone! CV mean F1 = {np.mean(fold_scores):.4f}")

if __name__ == "__main__":
    main()

MILK10k 2025  |  albumentations 2.0.8
Device: cuda
  GPU 0: Tesla T4  15.6 GB
  GPU 1: Tesla T4  15.6 GB

Loading CSVs...
  metadata (10480, 17)  groundtruth (5240, 12)  supplement (10480, 4)
  merged train_df: (5240, 31)
  test_df (deduped): (479, 17)

Preprocessing metadata...
  n_sites embedding: 10

Train: 5240  Test: 479
Class distribution:
  AKIEC     :   303  (5.8%)
  BCC       :  2522  (48.1%)
  BEN_OTH   :    44  (0.8%)
  BKL       :   544  (10.4%)
  DF        :    52  (1.0%)
  INF       :    50  (1.0%)
  MAL_OTH   :     9  (0.2%)
  MEL       :   450  (8.6%)
  NV        :   746  (14.2%)
  SCCKA     :   473  (9.0%)
  VASC      :    47  (0.9%)

Cross-Validation

  FOLD 1/3
  train=3493  val=1747


model.safetensors:   0%|          | 0.00/218M [00:00<?, ?B/s]

  DataParallel across 2 GPUs


Epoch 1 Train:   0%|          | 0/291 [00:00<?, ?it/s]

Validate:   0%|          | 0/73 [00:00<?, ?it/s]

...

  Loading fold 2...


Predict:   0%|          | 0/20 [00:00<?, ?it/s]

  Loading fold 3...


Predict:   0%|          | 0/20 [00:00<?, ?it/s]


Submission → /kaggle/working/submission.csv  shape=(479, 12)
       lesion     AKIEC       BCC   BEN_OTH       BKL        DF       INF  \
0  IL_0006205  0.276595  0.068357  0.010370  0.177553  0.001724  0.002224   
1  IL_0025400  0.988353  0.005874  0.002538  0.494986  0.001891  0.001555   
2  IL_0039001  0.014465  0.012507  0.116038  0.088789  0.031988  0.024705   
3  IL_0046799  0.005543  0.007656  0.000717  0.028709  0.001794  0.000635   
4  IL_0054262  0.005272  0.008735  0.006490  0.036683  0.912260  0.171712   

    MAL_OTH       MEL        NV     SCCKA      VASC  
0  0.001629  0.020439  0.007141  0.975739  0.015687  
1  0.000600  0.004016  0.002005  0.005209  0.001133  
2  0.003984  0.060148  0.947828  0.003444  0.029205  
3  0.000911  0.004930  0.001088  0.997652  0.001544  
4  0.030542  0.023482  0.114656  0.008236  0.022431  

Done! CV mean F1 = 0.4941
